In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('cs-training.csv')
df = df.drop('Unnamed: 0', axis=1)

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nClass distribution:")
print(df['SeriousDlqin2yrs'].value_counts())
print("\nMissing values:")
print(df.isnull().sum())

Shape: (150000, 11)

Columns: ['SeriousDlqin2yrs', 'RevolvingUtilizationOfUnsecuredLines', 'age', 'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 'MonthlyIncome', 'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfDependents']

Class distribution:
SeriousDlqin2yrs
0    139974
1     10026
Name: count, dtype: int64

Missing values:
SeriousDlqin2yrs                            0
RevolvingUtilizationOfUnsecuredLines        0
age                                         0
NumberOfTime30-59DaysPastDueNotWorse        0
DebtRatio                                   0
MonthlyIncome                           29731
NumberOfOpenCreditLinesAndLoans             0
NumberOfTimes90DaysLate                     0
NumberRealEstateLoansOrLines                0
NumberOfTime60-89DaysPastDueNotWorse        0
NumberOfDependents                       3924
dtype: int64


In [3]:
# Fix missing values
df['MonthlyIncome'] = df['MonthlyIncome'].fillna(df['MonthlyIncome'].median())
df['NumberOfDependents'] = df['NumberOfDependents'].fillna(0)

# Remove outliers — age 0 makes no sense
df = df[df['age'] > 0]

print("Missing values after cleaning:")
print(df.isnull().sum())
print("\nShape after cleaning:", df.shape)

Missing values after cleaning:
SeriousDlqin2yrs                        0
RevolvingUtilizationOfUnsecuredLines    0
age                                     0
NumberOfTime30-59DaysPastDueNotWorse    0
DebtRatio                               0
MonthlyIncome                           0
NumberOfOpenCreditLinesAndLoans         0
NumberOfTimes90DaysLate                 0
NumberRealEstateLoansOrLines            0
NumberOfTime60-89DaysPastDueNotWorse    0
NumberOfDependents                      0
dtype: int64

Shape after cleaning: (149999, 11)


In [4]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

X = df.drop('SeriousDlqin2yrs', axis=1)
y = df['SeriousDlqin2yrs']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Before SMOTE:")
print(y_train.value_counts())

sm = SMOTE(random_state=42)
X_res, y_res = sm.fit_resample(X_train, y_train)

print("\nAfter SMOTE:")
print(y_res.value_counts())

Before SMOTE:
SeriousDlqin2yrs
0    111978
1      8021
Name: count, dtype: int64

After SMOTE:
SeriousDlqin2yrs
0    111978
1    111978
Name: count, dtype: int64


In [6]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, classification_report

model = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    eval_metric='auc',
    random_state=42
)

model.fit(X_res, y_res)

y_pred_proba = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)

print("AUC-ROC:", round(roc_auc_score(y_test, y_pred_proba), 4))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

AUC-ROC: 0.8377

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.89      0.93     27995
           1       0.27      0.55      0.36      2005

    accuracy                           0.87     30000
   macro avg       0.62      0.72      0.65     30000
weighted avg       0.92      0.87      0.89     30000



In [7]:
model_tuned = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    gamma=1,
    eval_metric='auc',
    random_state=42
)

model_tuned.fit(X_res, y_res)

y_pred_proba_tuned = model_tuned.predict_proba(X_test)[:, 1]
y_pred_tuned = model_tuned.predict(X_test)

print("Tuned AUC-ROC:", round(roc_auc_score(y_test, y_pred_proba_tuned), 4))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_tuned))

Tuned AUC-ROC: 0.8374

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.90      0.93     27995
           1       0.28      0.54      0.37      2005

    accuracy                           0.87     30000
   macro avg       0.62      0.72      0.65     30000
weighted avg       0.92      0.87      0.89     30000



In [4]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

# Load & clean
df = pd.read_csv("cs-training.csv")
df = df.rename(columns={"SeriousDlqin2yrs": "target"})
df.drop(columns=["Unnamed: 0"], errors="ignore", inplace=True)

df["MonthlyIncome"] = df["MonthlyIncome"].fillna(df["MonthlyIncome"].median())
df["NumberOfDependents"] = df["NumberOfDependents"].fillna(0)
df = df[df["age"] > 0]

X = df.drop(columns=["target"])
y = df["target"]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# SMOTE on training set only
sm = SMOTE(random_state=42)
X_train_bal, y_train_bal = sm.fit_resample(X_train, y_train)
print(f"After SMOTE — class counts: {dict(zip(*np.unique(y_train_bal, return_counts=True)))}")

# Tuned XGBoost
model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=1,
    eval_metric="auc",
    early_stopping_rounds=20,
    random_state=42,
    n_jobs=-1,
)

model.fit(
    X_train_bal, y_train_bal,
    eval_set=[(X_test, y_test)],
    verbose=50,
)

# Evaluate
y_proba = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_proba)
print(f"\nAUC-ROC: {auc:.4f}  (target > 0.86)")
print(classification_report(y_test, model.predict(X_test)))

if auc < 0.86:
    print("AUC below target — consider increasing n_estimators or lowering learning_rate")

# Save model
with open("../models/credit_model.pkl", "wb") as f:
    pickle.dump(model, f)

with open("../models/feature_names.pkl", "wb") as f:
    pickle.dump(list(X.columns), f)

print("\nSaved → models/credit_model.pkl")
print("Saved → models/feature_names.pkl")

After SMOTE — class counts: {np.int64(0): np.int64(111978), np.int64(1): np.int64(111978)}
[0]	validation_0-auc:0.77128
[50]	validation_0-auc:0.84132
[92]	validation_0-auc:0.84279

AUC-ROC: 0.8432  (target > 0.86)
              precision    recall  f1-score   support

           0       0.97      0.87      0.92     27995
           1       0.26      0.62      0.36      2005

    accuracy                           0.85     30000
   macro avg       0.61      0.75      0.64     30000
weighted avg       0.92      0.85      0.88     30000

AUC below target — consider increasing n_estimators or lowering learning_rate

Saved → models/credit_model.pkl
Saved → models/feature_names.pkl


In [2]:
import os
print(os.getcwd())

/Users/parth/IdeaProjects/LoanLens/ml-service/notebooks


In [3]:
import os
print(os.listdir('.'))

['train_model.ipynb', '.gitkeep', 'ml-service', '.ipynb_checkpoints']


In [5]:
model_v2 = XGBClassifier(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.02,
    subsample=0.8,
    colsample_bytree=0.7,
    min_child_weight=3,
    gamma=0.5,
    reg_alpha=0.1,
    reg_lambda=1.5,
    eval_metric="auc",
    early_stopping_rounds=30,
    random_state=42,
    n_jobs=-1,
)

model_v2.fit(
    X_train_bal, y_train_bal,
    eval_set=[(X_test, y_test)],
    verbose=50,
)

y_proba_v2 = model_v2.predict_proba(X_test)[:, 1]
auc_v2 = roc_auc_score(y_test, y_proba_v2)
print(f"\nAUC-ROC v2: {auc_v2:.4f}")
print(classification_report(y_test, model_v2.predict(X_test)))

if auc_v2 > 0.86:
    with open("../models/credit_model.pkl", "wb") as f:
        pickle.dump(model_v2, f)
    print("New model saved — AUC target reached!")

[0]	validation_0-auc:0.76204
[47]	validation_0-auc:0.83680

AUC-ROC v2: 0.8405
              precision    recall  f1-score   support

           0       0.97      0.82      0.89     27995
           1       0.21      0.69      0.33      2005

    accuracy                           0.81     30000
   macro avg       0.59      0.76      0.61     30000
weighted avg       0.92      0.81      0.85     30000



In [3]:
# Feature engineering
df_feat = df.copy()

# Total delinquencies across all time windows
df_feat['total_late'] = (df_feat['NumberOfTime30-59DaysPastDueNotWorse'] + 
                          df_feat['NumberOfTime60-89DaysPastDueNotWorse'] + 
                          df_feat['NumberOfTimes90DaysLate'])

# Debt burden per dependent
df_feat['debt_per_dependent'] = df_feat['DebtRatio'] / (df_feat['NumberOfDependents'] + 1)

# Credit utilization per open line
df_feat['util_per_credit_line'] = (df_feat['RevolvingUtilizationOfUnsecuredLines'] / 
                                    (df_feat['NumberOfOpenCreditLinesAndLoans'] + 1))

# Monthly income to debt ratio
df_feat['income_debt_ratio'] = df_feat['MonthlyIncome'] / (df_feat['DebtRatio'] + 1)

# Age groups
df_feat['age_group'] = pd.cut(df_feat['age'], 
                               bins=[0, 30, 45, 60, 150], 
                               labels=[0, 1, 2, 3]).astype(float).fillna(0).astype(int)

X2 = df_feat.drop(columns=["target"])
y2 = df_feat["target"]

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42, stratify=y2
)

sm2 = SMOTE(random_state=42)
X2_train_bal, y2_train_bal = sm2.fit_resample(X2_train, y2_train)

model_v3 = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.02,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    gamma=0.3,
    eval_metric="auc",
    early_stopping_rounds=30,
    random_state=42,
    n_jobs=-1,
)

model_v3.fit(
    X2_train_bal, y2_train_bal,
    eval_set=[(X2_test, y2_test)],
    verbose=50,
)

y_proba_v3 = model_v3.predict_proba(X2_test)[:, 1]
auc_v3 = roc_auc_score(y2_test, y_proba_v3)
print(f"\nAUC-ROC v3 (with feature engineering): {auc_v3:.4f}")
print(classification_report(y2_test, model_v3.predict(X2_test)))

if auc_v3 > 0.86:
    with open("../models/credit_model.pkl", "wb") as f:
        pickle.dump(model_v3, f)
    with open("../models/feature_names.pkl", "wb") as f:
        pickle.dump(list(X2.columns), f)
    print("New model saved — AUC target reached!")
else:
    print(f"AUC is {auc_v3:.4f} — saving anyway as best model so far")
    with open("../models/credit_model.pkl", "wb") as f:
        pickle.dump(model_v3, f)
    with open("../models/feature_names.pkl", "wb") as f:
        pickle.dump(list(X2.columns), f)

[0]	validation_0-auc:0.84113
[50]	validation_0-auc:0.84940
[100]	validation_0-auc:0.85244
[150]	validation_0-auc:0.85317
[169]	validation_0-auc:0.85292

AUC-ROC v3 (with feature engineering): 0.8534
              precision    recall  f1-score   support

           0       0.97      0.90      0.93     27995
           1       0.30      0.60      0.40      2005

    accuracy                           0.88     30000
   macro avg       0.63      0.75      0.67     30000
weighted avg       0.92      0.88      0.90     30000

AUC is 0.8534 — saving anyway as best model so far


In [2]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

# Load & clean
df = pd.read_csv("cs-training.csv")
df = df.rename(columns={"SeriousDlqin2yrs": "target"})
df.drop(columns=["Unnamed: 0"], errors="ignore", inplace=True)

df["MonthlyIncome"] = df["MonthlyIncome"].fillna(df["MonthlyIncome"].median())
df["NumberOfDependents"] = df["NumberOfDependents"].fillna(0)
df = df[df["age"] > 0]

print("Data loaded:", df.shape)

Data loaded: (149999, 11)


In [4]:
# Cap outliers that are hurting the model
df_feat2 = df_feat.copy()

df_feat2['RevolvingUtilizationOfUnsecuredLines'] = df_feat2['RevolvingUtilizationOfUnsecuredLines'].clip(0, 1)
df_feat2['DebtRatio'] = df_feat2['DebtRatio'].clip(0, 5)
df_feat2['MonthlyIncome'] = df_feat2['MonthlyIncome'].clip(0, 50000)
df_feat2['NumberOfTime30-59DaysPastDueNotWorse'] = df_feat2['NumberOfTime30-59DaysPastDueNotWorse'].clip(0, 10)
df_feat2['NumberOfTime60-89DaysPastDueNotWorse'] = df_feat2['NumberOfTime60-89DaysPastDueNotWorse'].clip(0, 10)
df_feat2['NumberOfTimes90DaysLate'] = df_feat2['NumberOfTimes90DaysLate'].clip(0, 10)
df_feat2['total_late'] = df_feat2['total_late'].clip(0, 20)

X3 = df_feat2.drop(columns=["target"])
y3 = df_feat2["target"]

X3_train, X3_test, y3_train, y3_test = train_test_split(
    X3, y3, test_size=0.2, random_state=42, stratify=y3
)

sm3 = SMOTE(random_state=42)
X3_train_bal, y3_train_bal = sm3.fit_resample(X3_train, y3_train)

model_v4 = XGBClassifier(
    n_estimators=600,
    max_depth=6,
    learning_rate=0.01,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    gamma=0.3,
    reg_alpha=0.05,
    reg_lambda=1.0,
    eval_metric="auc",
    early_stopping_rounds=40,
    random_state=42,
    n_jobs=-1,
)

model_v4.fit(
    X3_train_bal, y3_train_bal,
    eval_set=[(X3_test, y3_test)],
    verbose=100,
)

y_proba_v4 = model_v4.predict_proba(X3_test)[:, 1]
auc_v4 = roc_auc_score(y3_test, y_proba_v4)
print(f"\nAUC-ROC v4 (capped outliers): {auc_v4:.4f}")
print(classification_report(y3_test, model_v4.predict(X3_test)))

with open("../models/credit_model.pkl", "wb") as f:
    pickle.dump(model_v4, f)
with open("../models/feature_names.pkl", "wb") as f:
    pickle.dump(list(X3.columns), f)
print("Model saved.")

[0]	validation_0-auc:0.83994
[43]	validation_0-auc:0.84582

AUC-ROC v4 (capped outliers): 0.8469
              precision    recall  f1-score   support

           0       0.97      0.84      0.90     27995
           1       0.24      0.69      0.35      2005

    accuracy                           0.83     30000
   macro avg       0.60      0.76      0.63     30000
weighted avg       0.92      0.83      0.87     30000

Model saved.


In [5]:
# Save v3 as the final model
with open("../models/credit_model.pkl", "wb") as f:
    pickle.dump(model_v3, f)
with open("../models/feature_names.pkl", "wb") as f:
    pickle.dump(list(X2.columns), f)

print("Final model saved — AUC-ROC: 0.8534")
print("Features:", list(X2.columns))

Final model saved — AUC-ROC: 0.8534
Features: ['RevolvingUtilizationOfUnsecuredLines', 'age', 'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 'MonthlyIncome', 'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfDependents', 'total_late', 'debt_per_dependent', 'util_per_credit_line', 'income_debt_ratio', 'age_group']
